# CCFULL-ROM parameter analysis

Reads one run of `python -m emulator.run_rom` (set `RUN` below) and ranks its
parameter points by

$$
\chi^2(\boldsymbol\theta)=\sum_i
\left[\frac{\sigma_{\rm ROM}(E_i;\boldsymbol\theta)
-\sigma_{\rm exp}(E_i)}{\Delta\sigma_i}\right]^2,
$$

with the upper or lower experimental uncertainty according to the sign of the
residual. The excitation function and the barrier distribution are shown at
the best point.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The run to analyze: a directory of emulator/outputs.
RUN = Path("outputs/20261104_001409_553140_16O_154Sm_Basis_300")

data = pd.read_csv(RUN / "cross_sections.csv")
reaction = json.loads((RUN / "run_summary.json").read_text())["reaction_system"]
parameters = ["beta2", "beta4", "beta6", "v0", "r0", "a0"]

# chi^2/N of every parameter point against the measured cross sections
measured = data.dropna(subset=["experimental_cross_section_mb"])
residual = measured["emulator_cross_section_mb"] - measured["experimental_cross_section_mb"]
error = np.where(residual >= 0, measured["experimental_uncertainty_upper_mb"],
                 measured["experimental_uncertainty_lower_mb"])
chi2 = ((residual / error) ** 2).groupby(measured["point_index"]).mean()

best = chi2.idxmin()
curve = data[data["point_index"] == best].sort_values("energy_cm_mev")
print(f"{reaction}: {len(chi2)} parameter points; best point {best}, chi2/N = {chi2[best]:.6g}")
curve[parameters].iloc[0]

## Excitation function at the best point

In [ ]:
E = curve["energy_cm_mev"]
plt.errorbar(E, curve["experimental_cross_section_mb"],
             yerr=[curve["experimental_uncertainty_lower_mb"], curve["experimental_uncertainty_upper_mb"]],
             fmt="ko", mfc="none", label="Experiment")
plt.plot(E, curve["emulator_cross_section_mb"], "b-", label="CCFULL-ROM")
if "exact_numerov_cross_section_mb" in curve:
    plt.plot(E, curve["exact_numerov_cross_section_mb"], "r--", label="Exact Numerov")
plt.yscale("log")
plt.xlabel("$E_{cm}$ (MeV)")
plt.ylabel(r"$\sigma_{fus}$ (mb)")
plt.title(f"{reaction}, best point {best}")
plt.legend()
plt.show()

## Barrier distribution at the best point

$$
D(E)=\frac{d^2[E\sigma(E)]}{dE^2}\approx
\frac{F(E+\Delta E)-2F(E)+F(E-\Delta E)}{\Delta E^2},
\qquad F=E\sigma .
$$

$F(E\pm\Delta E)$ is interpolated linearly between the energies of the run,
and energies without a complete stencil are left out. The experimental error
is propagated from the mean of the lower and upper uncertainties, assuming
independent points. `second_derivative` follows
`34S+168Er/analysis/Barrier_distribution.ipynb`.

In [ ]:
def second_derivative(E, sigma, sigma_error, dE):
    """D(E) and its error; NaN where E - dE or E + dE lies outside the energies."""
    F, F_error = E * sigma, E * sigma_error
    D, D_error = np.full(len(E), np.nan), np.full(len(E), np.nan)

    def add(weights, target, scale):
        # F at target, linearly interpolated, as weights on the given energies
        right = np.searchsorted(E, target)
        if right < len(E) and abs(E[right] - target) <= 1.0e-10:
            weights[right] += scale
            return
        fraction = (target - E[right - 1]) / (E[right] - E[right - 1])
        weights[right - 1] += scale * (1.0 - fraction)
        weights[right] += scale * fraction

    for i in range(len(E)):
        if E[i] - dE < E[0] or E[i] + dE > E[-1]:
            continue
        weights = np.zeros(len(E))
        add(weights, E[i] - dE, 1.0 / dE**2)
        weights[i] += -2.0 / dE**2
        add(weights, E[i] + dE, 1.0 / dE**2)
        D[i] = weights @ F
        D_error[i] = np.sqrt(np.sum((weights * F_error) ** 2))
    return D, D_error


dE = 2 * (154.0 / (16.0 + 154.0))      # MeV: a 2 MeV step in the laboratory frame
E = curve["energy_cm_mev"].to_numpy()
error = 0.5 * (curve["experimental_uncertainty_lower_mb"] + curve["experimental_uncertainty_upper_mb"]).to_numpy()
D_exp, D_exp_error = second_derivative(E, curve["experimental_cross_section_mb"].to_numpy(), error, dE)
D_rom, _ = second_derivative(E, curve["emulator_cross_section_mb"].to_numpy(), 0 * E, dE)
used = np.isfinite(D_exp) & (D_exp_error > 0)
print(f"CCFULL-ROM: chi2_D/N = {np.mean(((D_rom - D_exp) / D_exp_error)[used] ** 2):.3g} over {used.sum()} energies")

plt.errorbar(E, D_exp, yerr=D_exp_error, fmt="ko", mfc="none", label="Experiment")
plt.plot(E, D_rom, "b-", label="CCFULL-ROM")
if "exact_numerov_cross_section_mb" in curve:
    D_exact, _ = second_derivative(E, curve["exact_numerov_cross_section_mb"].to_numpy(), 0 * E, dE)
    plt.plot(E, D_exact, "r--", label="Exact Numerov")
    print(f"Exact Numerov: chi2_D/N = {np.mean(((D_exact - D_exp) / D_exp_error)[used] ** 2):.3g}; "
          f"max |D_ROM - D_exact| = {np.nanmax(np.abs(D_rom - D_exact)):.3g} mb/MeV")
plt.xlim(50, 75)
plt.ylim(-100, 750)
plt.xlabel("$E_{cm}$ (MeV)")
plt.ylabel(r"$d^2(E\sigma)/dE^2$ (mb/MeV)")
plt.title(rf"{reaction}, best point {best}, $\Delta E$ = {dE:.3g} MeV")
plt.legend(loc="upper left")
plt.show()